# Two-layer network on imbalanced data with custom losses

**Goal:** show how the choice of loss changes what a small network learns when the positive class is rare (~2%).

**Losses compared**
1. Plain cross-entropy (baseline)
2. Weighted cross-entropy (inverse-frequency weights)
3. Focal loss (custom)
4. Class-balanced focal loss (custom, effective-number weights)

**Why accuracy is useless here:** predicting 'negative' for everything gives ~98% accuracy. We evaluate with PR-AUC, best-threshold F1, and recall at a fixed false-positive rate.

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, roc_curve, precision_recall_curve
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Device: cpu


## 1. Data

Synthetic tabular data with a ~2% positive rate, a non-linear structure (2 clusters per class), and some class overlap so the problem is not trivial. Split is stratified into train / validation / test so every split keeps the same imbalance.

In [2]:
DATA_SEED = 42

X, y = make_classification(
    n_samples=60000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_clusters_per_class=2,
    weights=[0.98, 0.02],
    class_sep=0.8,
    flip_y=0.0,
    random_state=DATA_SEED,
)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=DATA_SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=DATA_SEED
)

scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = (scaler.transform(a) for a in (X_train, X_val, X_test))

def to_t(a, dtype):
    return torch.tensor(a, dtype=dtype)

Xtr, Xva, Xte = (to_t(a, torch.float32) for a in (X_train, X_val, X_test))
ytr, yva, yte = (to_t(a, torch.long) for a in (y_train, y_val, y_test))

class_counts = np.bincount(y_train, minlength=2)
print('Train class counts [neg, pos]:', class_counts)
print('Positive rate: train=%.3f val=%.3f test=%.3f' % (y_train.mean(), y_val.mean(), y_test.mean()))

Train class counts [neg, pos]: [35280   720]
Positive rate: train=0.020 val=0.020 test=0.020


## 2. Model: a 2-layer network

`input -> Linear -> ReLU -> Linear -> 2 logits`. Two weight matrices, one hidden layer.

In [3]:
class TwoLayerNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, n_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

## 3. Losses

**Why plain CE struggles:** with 98% negatives, the summed gradient is dominated by easy negatives. The model quickly learns 'predict negative' and the rare positives contribute little signal.

**Focal loss** fixes the *easy-example flood*:

`FL = -alpha_t * (1 - p_t)^gamma * log(p_t)`

- `p_t` is the predicted probability of the true class.
- `(1 - p_t)^gamma` shrinks the loss of well-classified examples (mostly negatives) toward zero, so training focuses on hard examples.
- `alpha_t` is an optional per-class weight.

**Class-balanced weights** (Cui et al., 2019) use the *effective number of samples* `E_n = (1 - beta^n) / (1 - beta)`, giving weights `w_c ~ 1 / E_{n_c}`. With `beta -> 1` this approaches inverse frequency, and with `beta = 0` it gives uniform weights. This is gentler than raw inverse frequency for extreme imbalance.

In [4]:
def class_balanced_weights(counts, beta=0.999):
    '''Effective-number weights, normalised so they sum to the number of classes.'''
    counts = np.asarray(counts, dtype=np.float64)
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    w = 1.0 / effective_num
    w = w / w.sum() * len(counts)
    return torch.tensor(w, dtype=torch.float32)


def inverse_freq_weights(counts):
    counts = np.asarray(counts, dtype=np.float64)
    w = counts.sum() / (len(counts) * counts)
    return torch.tensor(w, dtype=torch.float32)


class FocalLoss(nn.Module):
    '''Multi-class focal loss with optional per-class alpha weights.

    FL = -alpha_t * (1 - p_t)^gamma * log(p_t)
    '''
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer('alpha', alpha.clone().detach())
        else:
            self.alpha = None

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)                       # stable log p
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # log p_t
        pt = log_pt.exp()
        loss = -((1.0 - pt) ** self.gamma) * log_pt
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


class WeightedCE(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.register_buffer('weight', weight.clone().detach())

    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets, weight=self.weight)


class PlainCE(nn.Module):
    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets)


# Registry: name -> factory (so each run gets a fresh loss module)
LOSSES = {
    'CE (baseline)': lambda: PlainCE(),
    'Weighted CE': lambda: WeightedCE(inverse_freq_weights(class_counts)),
    'Focal (gamma=2)': lambda: FocalLoss(gamma=2.0),
    'Class-balanced Focal': lambda: FocalLoss(
        gamma=2.0, alpha=class_balanced_weights(class_counts, beta=0.999)
    ),
}

print('Inverse-freq weights:', inverse_freq_weights(class_counts).tolist())
print('Class-balanced weights:', class_balanced_weights(class_counts).tolist())

Inverse-freq weights: [0.5102040767669678, 25.0]
Class-balanced weights: [0.6784924268722534, 1.3215075731277466]


## 4. Training and evaluation

Model selection uses **validation PR-AUC** (threshold-free, sensitive to the minority class). The decision threshold for F1 is chosen on the **validation** set and applied unchanged to the test set, so test numbers are not tuned on test data.

`recall@FPR` answers: *of all true positives, what fraction do we catch if we only tolerate 1% of negatives being flagged?* This is often the operational question in fraud-style problems.

In [5]:
@torch.no_grad()
def predict_proba(model, X, batch_size=4096):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        logits = model(X[i:i + batch_size].to(DEVICE))
        out.append(F.softmax(logits, dim=1)[:, 1].cpu())
    return torch.cat(out).numpy()


def best_f1_threshold(probs, y_true):
    p, r, t = precision_recall_curve(y_true, probs)
    f1 = 2 * p * r / (p + r + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))   # last PR point has no threshold
    return float(t[idx])


def f1_at(probs, y_true, thr):
    preds = probs >= thr
    tp = np.sum(preds & (y_true == 1))
    fp = np.sum(preds & (y_true == 0))
    fn = np.sum((~preds) & (y_true == 1))
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    return 2 * prec * rec / (prec + rec + 1e-12), prec, rec


def recall_at_fpr(probs, y_true, max_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, probs)
    return float(tpr[fpr <= max_fpr].max())


def train_one(loss_fn, seed, epochs=40, lr=1e-3, batch_size=512, hidden_dim=64):
    set_seed(seed)
    model = TwoLayerNet(Xtr.shape[1], hidden_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = loss_fn.to(DEVICE)

    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True, generator=g
    )

    best_ap, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        val_ap = average_precision_score(y_val, predict_proba(model, Xva))
        if val_ap > best_ap:
            best_ap, best_state = val_ap, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model


def evaluate_model(model):
    val_probs = predict_proba(model, Xva)
    test_probs = predict_proba(model, Xte)
    thr = best_f1_threshold(val_probs, y_val)          # chosen on validation only
    f1, prec, rec = f1_at(test_probs, y_test, thr)
    return {
        'PR-AUC': average_precision_score(y_test, test_probs),
        'ROC-AUC': roc_auc_score(y_test, test_probs),
        'F1': f1,
        'Precision': prec,
        'Recall': rec,
        'Recall@1%FPR': recall_at_fpr(test_probs, y_test, 0.01),
    }

## 5. Run the comparison

Same data, architecture, optimizer, and epochs for every loss. Multiple seeds, so one lucky initialisation does not decide the ranking.

In [6]:
SEEDS = [0, 1, 2, 3, 4]
rows = []

for name, make_loss in LOSSES.items():
    for seed in SEEDS:
        model = train_one(make_loss(), seed)
        res = evaluate_model(model)
        res.update({'Loss': name, 'Seed': seed})
        rows.append(res)
        print(f"{name:22s} seed={seed}  PR-AUC={res['PR-AUC']:.4f}  F1={res['F1']:.4f}")

results = pd.DataFrame(rows)
summary = results.groupby('Loss').agg(['mean', 'std']).drop(columns='Seed')
summary = summary.loc[list(LOSSES.keys())]
summary.round(4)

CE (baseline)          seed=0  PR-AUC=0.7007  F1=0.6652
CE (baseline)          seed=1  PR-AUC=0.6936  F1=0.6696
CE (baseline)          seed=2  PR-AUC=0.6888  F1=0.6504
CE (baseline)          seed=3  PR-AUC=0.6975  F1=0.6682
CE (baseline)          seed=4  PR-AUC=0.6915  F1=0.6535
Weighted CE            seed=0  PR-AUC=0.6799  F1=0.6473
Weighted CE            seed=1  PR-AUC=0.7003  F1=0.6410
Weighted CE            seed=2  PR-AUC=0.6802  F1=0.6310
Weighted CE            seed=3  PR-AUC=0.6921  F1=0.6275
Weighted CE            seed=4  PR-AUC=0.6799  F1=0.6316
Focal (gamma=2)        seed=0  PR-AUC=0.7460  F1=0.6833
Focal (gamma=2)        seed=1  PR-AUC=0.7520  F1=0.6954
Focal (gamma=2)        seed=2  PR-AUC=0.7184  F1=0.6780
Focal (gamma=2)        seed=3  PR-AUC=0.7502  F1=0.6912
Focal (gamma=2)        seed=4  PR-AUC=0.7430  F1=0.7101
Class-balanced Focal   seed=0  PR-AUC=0.7464  F1=0.6818
Class-balanced Focal   seed=1  PR-AUC=0.7513  F1=0.6801
Class-balanced Focal   seed=2  PR-AUC=0.7236  F1

PR-AUC         ROC-AUC              F1          \
                        mean     std    mean     std    mean     std   
Loss                                                                   
CE (baseline)         0.6944  0.0047  0.9676  0.0008  0.6614  0.0088   
Weighted CE           0.6865  0.0093  0.9718  0.0006  0.6357  0.0082   
Focal (gamma=2)       0.7419  0.0136  0.9749  0.0017  0.6916  0.0124   
Class-balanced Focal  0.7438  0.0116  0.9760  0.0015  0.6816  0.0089   

                     Precision          Recall         Recall@1%FPR          
                          mean     std    mean     std         mean     std  
Loss                                                                         
CE (baseline)           0.7253  0.0366  0.6100  0.0314       0.6683  0.0086  
Weighted CE             0.7753  0.0510  0.5408  0.0251       0.6675  0.0095  
Focal (gamma=2)         0.8327  0.0177  0.5917  0.0167       0.7192  0.0171  
Class-balanced Focal    0.8558  0.0165  0.5667  0.0144       0.7183  0.0100

## 6. How to read the results

- **Compare PR-AUC first.** It is threshold-free and reflects ranking quality on the minority class. Check whether differences exceed the seed-to-seed std before claiming a winner.
- **F1 / Precision / Recall** use a threshold tuned on validation. Weighted losses typically shift the optimal threshold, which is why we tune it per model rather than fixing 0.5.
- **Recall@1%FPR** is the operational metric: the detection rate at a fixed false-alarm budget.
- **Expect modest gaps.** On moderately separable data, plain CE plus a tuned threshold is often competitive. The larger wins from focal and class-balanced losses show up with more extreme imbalance (0.1% or lower) or harder overlap. Try changing `weights=[0.999, 0.001]` in the data cell.

### Things to try
| Experiment | What it teaches |
|---|---|
| Sweep `gamma` in {0, 0.5, 1, 2, 5} | gamma=0 recovers (weighted) CE; large gamma can underfit |
| Sweep `beta` in {0.9, 0.99, 0.999, 0.9999} | how much re-weighting is too much |
| Lower the positive rate to 0.1% | where baseline CE breaks down |
| Compare against `WeightedRandomSampler` | re-sampling vs re-weighting |

### Caveats
- **Calibration:** weighted and focal losses shift predicted probabilities away from true frequencies. Use the scores for ranking and thresholding. If you need real probabilities, recalibrate (Platt or isotonic) on validation data.
- **Loss value is not comparable across losses.** Never pick the model by comparing raw loss values between CE and focal. Use the validation PR-AUC as done here.